# Establishing correctness by comparing outputs

This notebook tests the correctness of the different implementations by comparing outputs to a simple exhaustive loop search or across implmentations.

Note that the notebook can take a couple of minutes of run time.

We furthermore rerun every algorithm on an independently, randomly reordered copy of the same input, remap the returned indices back to the original order, and check the result still matches. This tests that outputs do not depend on input order.

In [1]:
n_repeat = 1

## Installation (< 2min)

In [2]:
try:
    from google.colab import files
    colab = True
except ImportError:
    colab = False

if colab and not os.path.exists("XT-neighbor"):
    !git clone https://github.com/heartnetkung/XT-neighbor.git
    repo_path = "XT-neighbor/"
else:
    repo_path = "../"

compile XTNeighbor-streaming

In [3]:
! mkdir -p {repo_path}/xtneighbor_streaming/build
! cd {repo_path}/xtneighbor_streaming/build; cmake -S .. -B .;make

-- Configuring done (0.1s)
-- Generating done (0.0s)
-- Build files have been written to: /home/andreas/repos/XT-neighbor/xtneighbor_streaming/build


[ 50%] Building CUDA object CMakeFiles/xt_neighbor.dir/src/cli.cu.o


[100%] Linking CUDA executable xt_neighbor


[100%] Built target xt_neighbor


compile XTNeighbor

In [4]:
! mkdir -p {repo_path}/xtneighbor/build
! cd {repo_path}/xtneighbor/build; cmake -S .. -B .;make

-- Configuring done (0.0s)
-- Generating done (0.0s)
-- Build files have been written to: /home/andreas/repos/XT-neighbor/xtneighbor/build
[100%] Built target xt_neighbor


## Setup and data import

In [5]:
from os import path
import numpy as np
import pandas as pd
import seaborn as sns
import time
import subprocess
import re
import random
from pyrepseq import levenshtein_distance
import pyrepseq
import symscan

In [6]:
import benchutils as bu

bu.describe_env()

{'colab': False,
 'platform': 'Linux-6.8.0-136-generic-x86_64-with-glibc2.39',
 'python': '3.12.13',
 'git_sha': '7e33b83',
 'cpu_model': '13th Gen Intel(R) Core(TM) i9-13900',
 'n_cpus_total': 32,
 'affinity': [0, 2, 4, 6, 8, 10, 12, 14],
 'n_cpus_visible': 8,
 'governor': 'performance',
 'no_turbo': '1',
 'gpu': {'name': 'NVIDIA GeForce RTX 4090',
  'memory_total': '24564 MiB',
  'clocks_max_sm': '3105 MHz',
  'clocks_applications_gr': '[N/A]'},
 'thread_env': {'RAYON_NUM_THREADS': '8',
  'OMP_NUM_THREADS': '8',
  'MKL_NUM_THREADS': '8',
  'OPENBLAS_NUM_THREADS': '8',
  'NUMEXPR_NUM_THREADS': '8',
  'NUMBA_NUM_THREADS': '8',
  'OMP_PROC_BIND': None,
  'OMP_PLACES': None},
 'packages': {'symscan': '0.8.3',
  'pyrepseq': '1.6',
  'pybktree': '1.1',
  'rapidfuzz': '3.14.5',
  'pwseqdist': '0.6',
  'numba': '0.66.0',
  'numpy': '2.4.6',
  'scipy': '1.18.0',
  'pandas': '3.0.5'},
 'timeout_seconds': 100}

read in data

In [7]:
N_FILES=1

data = []
for i in range(1,N_FILES+1):
  data += pd.read_csv(f'{repo_path}/data/emerson{i}_dl.zip', compression='zip', header=0)['cdr3'].to_list()

print('first row:', data[0]);
print(f'len: {len(data):,}')

first row: CAAAAGGIAKNIQYF
len: 10,000,000


In [8]:
lengths = [len(s) for s in data]

Write files functions used for XTNeighbor and XTNeighbor-streaming respectively

In [9]:
!mkdir -p tmp

In [10]:
def writeFile(seqs):
  with open("tmp/input.txt","w") as file1:
    file1.writelines(seq+'\n' for seq in seqs)

def writeFile2(seqs):
  with open("tmp/input2.txt","w") as file1:
    file1.writelines(seq+'\n' for seq in (['cdr3']+seqs))

## Implementations

In [11]:
def xt_neighbor(seqs,threshold,_len): #verbose is ignored
  ! {repo_path}/xtneighbor/build/xt_neighbor -p "tmp/input.txt" -n "$_len" -d "$threshold" -o "tmp/xt_output.txt"
  return read_result('tmp/xt_output.txt')

def xt_neighbor_streaming(seqs,threshold,_len):
  ! {repo_path}/xtneighbor_streaming/build/xt_neighbor -i "tmp/input2.txt" -n "$_len" -d "$threshold" -o "tmp/xt_streaming_output.txt"
  return read_result('tmp/xt_streaming_output.txt')

def symdel(seqs,threshold,_len):
  return set(pyrepseq.symdel(seqs,max_edits=threshold, output_symmetric=False))

def run_symscan(seqs,threshold,_len):
  row, col, dists = symscan.get_neighbors_within(seqs, max_distance=threshold)
  return set(zip(row, col, dists))

def for_loop(seqs,threshold,_len):
  ans = set()
  for i in range(len(seqs)):
    for j in range(len(seqs)):
        if i >= j:
            continue
        dist = levenshtein_distance(seqs[j], seqs[i], score_cutoff=threshold)
        if dist <= threshold:
            ans.add((i, j, dist))
  return ans

def prepare(seqs):
  writeFile(seqs)
  writeFile2(seqs)

def read_result(filename):
  df = pd.read_csv(filename, sep=' ', header=None)
  row_set = set(df.itertuples(index=False, name=None))
  return row_set


In [12]:
def perform(subset, distance, algorithms, size):
    result=None
    for alg_name in algorithms:
        print('running algorithm:', alg_name)
        prepare(subset)
        new_result = algorithms[alg_name](subset, distance, size)
        if result is None:
            result = new_result
        elif result != new_result:
            raise Exception('comparison failed')
    return result


def perform_shuffled(subset, distance, algorithms, size, expected):
    perm = list(range(size))
    random.Random(f"{size}-{distance}-shuffle").shuffle(perm)
    shuffled = [subset[k] for k in perm]
    for alg_name in algorithms:
        print('running algorithm on shuffled input:', alg_name)
        prepare(shuffled)
        raw_result = algorithms[alg_name](shuffled,distance,size)
        new_result = {(min(perm[a], perm[b]), max(perm[a], perm[b]), d) for a, b, d in raw_result}
        if new_result != expected:
            raise Exception('shuffle comparison failed')


def run_exp(distance, size):
    for i in range(n_repeat):
        subset = random.Random(i).sample(data, size)
        result = perform(subset, distance, algorithms, size)
        perform_shuffled(subset, distance, algorithms, size, result)
    print('success!')

## Run on small datasets (< 2 min)

In [13]:
algorithms = {
    'for_loop':for_loop,
    'symdel':symdel,
    'symscan':run_symscan,
    #'xt':xt_neighbor,
    'xt_streaming':xt_neighbor_streaming,

}

In [14]:
run_exp(distance=1, size=5000)

running algorithm: for_loop


running algorithm: symdel


running algorithm: symscan
running algorithm: xt_streaming


running algorithm on shuffled input: for_loop


running algorithm on shuffled input: symdel


running algorithm on shuffled input: symscan
running algorithm on shuffled input: xt_streaming


success!


In [15]:
run_exp(distance=2, size=5000)

running algorithm: for_loop


running algorithm: symdel


running algorithm: symscan
running algorithm: xt_streaming


running algorithm on shuffled input: for_loop


running algorithm on shuffled input: symdel


running algorithm on shuffled input: symscan
running algorithm on shuffled input: xt_streaming


success!


In [16]:
run_exp(distance=3, size=5000)

running algorithm: for_loop


running algorithm: symdel


running algorithm: symscan
running algorithm: xt_streaming


running algorithm on shuffled input: for_loop


running algorithm on shuffled input: symdel


running algorithm on shuffled input: symscan
running algorithm on shuffled input: xt_streaming


success!


## Run on large datasets (< 15 min)

In [17]:
algorithms = {
    'xt_streaming':xt_neighbor_streaming,
    'symscan':run_symscan,
}

In [18]:
run_exp(distance=1, size=3_000_000)

running algorithm: xt_streaming


running algorithm: symscan


running algorithm on shuffled input: xt_streaming


running algorithm on shuffled input: symscan


success!


In [19]:
run_exp(distance=2, size=1_000_000)

running algorithm: xt_streaming


running algorithm: symscan


running algorithm on shuffled input: xt_streaming


running algorithm on shuffled input: symscan


success!


In [20]:
run_exp(distance=3, size=100_000)

running algorithm: xt_streaming


running algorithm: symscan


running algorithm on shuffled input: xt_streaming


running algorithm on shuffled input: symscan


success!


Shuffling test at very large scales (for xt_streaming only to save on computing time)

In [21]:
algorithms = {
    'xt_streaming':xt_neighbor_streaming,
}

In [22]:
run_exp(distance=1, size=10_000_000)

running algorithm: xt_streaming


running algorithm on shuffled input: xt_streaming


success!
